In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -U transformers

In [ ]:
import torch
import math
import random
import numpy as np
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorWithPadding, set_seed
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm.auto import tqdm
import re

In [ ]:
SEED = 42
BATCH_SIZE = 8
MAX_LENGTH = 512
STRIDE = 128
MLM_PROBABILITY = 0.15

In [ ]:
MODELS = [
    "dbmdz/bert-base-italian-xxl-cased",
    "linhd-postdata/alberti-bert-base-multilingual-cased",
    "mattiaferrarini/saba"
]
DATASET_PATH = "/content/drive/My Drive/italian_poems/italian_poems_test_rhymes.json"

In [ ]:
def fix_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

fix_seed(SEED)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

In [ ]:
for model_id in MODELS:
    print(f"\nEvaluating: {model_id}")
    fix_seed(SEED)

    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if not tokenizer.is_fast:
        print(f"Warning: {model_id} tokenizer is not 'fast'. Loading fast version.")
        tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

    model = AutoModelForMaskedLM.from_pretrained(model_id).to(device)
    model.eval()

    # Filter only by length
    def filter_long_samples(examples):
        tokenized = tokenizer(examples["text"], truncation=False, padding=False)
        return [len(ids) <= MAX_LENGTH for ids in tokenized["input_ids"]]

    filtered_dataset = dataset.filter(
        filter_long_samples,
        batched=True,
        desc=f"Filtering > {MAX_LENGTH} tokens"
    )
    print("Total poems for evaluation:", len(filtered_dataset))

    # Tokenization, masking and rhyme tagging
    def process_data(examples):
        tokenized = tokenizer(
            examples["text"],
            padding="max_length",
            max_length=MAX_LENGTH,
            truncation=True,
            return_offsets_mapping=True,
            return_special_tokens_mask=True
        )

        new_input_ids = []
        new_labels = []
        mask_group_ids = []  # Group ID to track sub-tokens of the masked word
        is_rhyming_sample = [] # 1 if poem has rhyme, 0 otherwise

        for i, text in enumerate(examples["text"]):
            # Determine if poem has rhymes
            rhyme_tags = examples["rhyme_tags"][i]
            has_rhyme = 0
            if rhyme_tags:
                last_tag = rhyme_tags[-1]
                # Check if last tag exists and appears at least twice (a rhyme pair exists)
                if last_tag is not None and rhyme_tags.count(last_tag) >= 2:
                    has_rhyme = 1
            is_rhyming_sample.append(has_rhyme)

            # Masking
            input_ids = tokenized["input_ids"][i]
            offsets = tokenized["offset_mapping"][i]
            special_mask = tokenized["special_tokens_mask"][i]

            labels = [-100] * len(input_ids)
            current_mask_group = [0] * len(input_ids)

            # Find last word
            matches = list(re.finditer(r"\b\w+\b", text))
            if matches:
                last_match = matches[-1]
                start_char, end_char = last_match.span()

                masked_at_least_one = False
                for idx, (tok_start, tok_end) in enumerate(offsets):
                    if special_mask[idx] or tok_start == tok_end:
                        continue

                    if tok_start < end_char and tok_end > start_char:
                        labels[idx] = input_ids[idx]
                        input_ids[idx] = tokenizer.mask_token_id
                        current_mask_group[idx] = 1
                        masked_at_least_one = True

                # Fallback
                if not masked_at_least_one:
                    for idx in range(len(input_ids)-1, -1, -1):
                        if input_ids[idx] != tokenizer.pad_token_id and not special_mask[idx]:
                            labels[idx] = input_ids[idx]
                            input_ids[idx] = tokenizer.mask_token_id
                            current_mask_group[idx] = 1
                            break

            new_input_ids.append(input_ids)
            new_labels.append(labels)
            mask_group_ids.append(current_mask_group)

        tokenized["input_ids"] = new_input_ids
        tokenized["labels"] = new_labels
        tokenized["mask_group_ids"] = mask_group_ids
        tokenized["is_rhyming"] = is_rhyming_sample
        return tokenized

    tokenized_datasets = filtered_dataset.map(
        process_data,
        batched=True,
        remove_columns=dataset.column_names,
        desc="Processing"
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    dataloader = DataLoader(
        tokenized_datasets,
        batch_size=BATCH_SIZE,
        collate_fn=data_collator
    )

    # Metrics containers
    stats = {
        "rhyme":     {"tok_cor": 0, "tok_tot": 0, "word_cor": 0, "word_tot": 0},
        "non_rhyme": {"tok_cor": 0, "tok_tot": 0, "word_cor": 0, "word_tot": 0}
    }

    for batch in tqdm(dataloader, desc="Eval"):
        # Extract metadata
        mask_groups = batch.pop("mask_group_ids").to(device) # Keep on GPU
        is_rhyming = batch.pop("is_rhyming").to(device)      # Keep on GPU

        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch["labels"]

        with torch.no_grad():
            outputs = model(**batch)
            predictions = torch.argmax(outputs.logits, dim=-1)

        # Evaluation
        correct_preds = (predictions == labels)
        relevant_tokens_mask = (mask_groups == 1)
        rhyme_rows = (is_rhyming == 1)
        non_rhyme_rows = (is_rhyming == 0)

        # Update token counts
        stats["rhyme"]["tok_tot"] += relevant_tokens_mask[rhyme_rows].sum().item()
        stats["rhyme"]["tok_cor"] += (correct_preds & relevant_tokens_mask)[rhyme_rows].sum().item()

        stats["non_rhyme"]["tok_tot"] += relevant_tokens_mask[non_rhyme_rows].sum().item()
        stats["non_rhyme"]["tok_cor"] += (correct_preds & relevant_tokens_mask)[non_rhyme_rows].sum().item()

        # Word-level stats
        incorrect_relevant = (~correct_preds & relevant_tokens_mask)

        has_relevant_tokens = relevant_tokens_mask.sum(dim=1) > 0
        word_has_error = incorrect_relevant.sum(dim=1) > 0
        word_is_correct = has_relevant_tokens & (~word_has_error)

        # Update word counts
        stats["rhyme"]["word_tot"] += has_relevant_tokens[rhyme_rows].sum().item()
        stats["rhyme"]["word_cor"] += word_is_correct[rhyme_rows].sum().item()

        stats["non_rhyme"]["word_tot"] += has_relevant_tokens[non_rhyme_rows].sum().item()
        stats["non_rhyme"]["word_cor"] += word_is_correct[non_rhyme_rows].sum().item()

    # Reporting
    print(f"\nResults for {model_id}:")
    print("=" * 60)

    # Broken-down stats
    for category_name, display_name in [("rhyme", "Rhyming Poems"), ("non_rhyme", "Non-Rhyming Poems")]:
        s = stats[category_name]
        print(f"Dataset: {display_name}")

        if s["tok_tot"] > 0:
            tok_acc = (s["tok_cor"] / s["tok_tot"]) * 100
            print(f"  Token Accuracy: {tok_acc:.2f}%  ({s['tok_cor']}/{s['tok_tot']})")
        else:
            print(f"  Token Accuracy: N/A (0 samples)")

        if s["word_tot"] > 0:
            word_acc = (s["word_cor"] / s["word_tot"]) * 100
            print(f"  Word Accuracy:  {word_acc:.2f}%  ({s['word_cor']}/{s['word_tot']})")
        else:
            print(f"  Word Accuracy:  N/A (0 samples)")
        print("-" * 60)

    # Overall stats
    overall_tok_cor = stats["rhyme"]["tok_cor"] + stats["non_rhyme"]["tok_cor"]
    overall_tok_tot = stats["rhyme"]["tok_tot"] + stats["non_rhyme"]["tok_tot"]
    overall_word_cor = stats["rhyme"]["word_cor"] + stats["non_rhyme"]["word_cor"]
    overall_word_tot = stats["rhyme"]["word_tot"] + stats["non_rhyme"]["word_tot"]

    print("Dataset: OVERALL (All Poems)")
    if overall_tok_tot > 0:
        overall_tok_acc = (overall_tok_cor / overall_tok_tot) * 100
        print(f"  Overall Token Accuracy: {overall_tok_acc:.2f}%  ({overall_tok_cor}/{overall_tok_tot})")
    else:
        print(f"  Overall Token Accuracy: N/A (0 samples)")

    if overall_word_tot > 0:
        overall_word_acc = (overall_word_cor / overall_word_tot) * 100
        print(f"  Overall Word Accuracy:  {overall_word_acc:.2f}%  ({overall_word_cor}/{overall_word_tot})")
    else:
        print(f"  Overall Word Accuracy:  N/A (0 samples)")
    print("=" * 60)

    del model
    torch.cuda.empty_cache()